In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import regex as re

html_content = requests.get(
    url="https://openaccess.thecvf.com/ICCV2025_workshops/menu"
).text
soup = BeautifulSoup(html_content, "html.parser")


In [2]:
base_url = "https://openaccess.thecvf.com"

In [3]:
regex_workshops = re.compile(r"^/ICCV2025_workshops/.*")

In [4]:
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<link href="/favicon.ico" rel="icon" type="image/png"/>
<title>ICCV 2025 Open Access Repository</title>
<link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.7/css/bootstrap.min.css" rel="stylesheet"/>
<script src="https://ajax.googleapis.com/ajax/libs/jquery/3.1.1/jquery.min.js" type="text/javascript"></script>
<script src="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.7/js/bootstrap.min.js" type="text/javascript"></script>
<script src="/static/jquery.js" type="text/javascript"></script>
<link href="/static/conf.css" rel="stylesheet" type="text/css"/>
</head>
<body>
<div id="header">
<div id="header_left">
<a href="https://iccv.thecvf.com/"><img alt="ICCV 2025" border="0" src="/img/iccv2025-logo.svg" width="175"/></a>
<a href="https://www.thecvf.com/"><img alt="CVF" border="0" height="112" src="/img/crop

In [5]:
workshops = soup.find_all(
        "a", attrs={"href": regex_workshops}
    )

print(workshops)
print(workshops[0].get("href"))

[<a href="/ICCV2025_workshops/ABAW">Affective &amp; Behavior Analysis in-the-wild</a>, <a href="/ICCV2025_workshops/CALIPOSE">Camera Calibration and Pose Estimation</a>, <a href="/ICCV2025_workshops/ILR+G">Instance-Level Recognition and Generation Workshop</a>, <a href="/ICCV2025_workshops/STREAM">Systematic Trust in AI Models: Ensuring Fairness, Reliability, Explainability, and Accountability in Machine Learning Frameworks</a>, <a href="/ICCV2025_workshops/DRL4Real">The 1st International Workshop and Challenge on Disentangled Representation Learning for Controllable Generation</a>, <a href="/ICCV2025_workshops/AI4VA">The 2nd AI for Visual Arts Workshop</a>, <a href="/ICCV2025_workshops/DataCV">The 4th DataCV Workshop and Challenge</a>, <a href="/ICCV2025_workshops/DeepID">The Challenge of Detecting Synthetic Manipulations in ID Documents</a>, <a href="/ICCV2025_workshops/CVPM">International Workshop on Computer Vision for Physiological Measurement</a>, <a href="/ICCV2025_workshops/BIS

In [6]:
all_titles = []
all_authors = []
all_links = []

for ws in workshops:
    if "latinx" in ws.get("href").lower():
        continue
    html_content = requests.get(
    url=f'https://openaccess.thecvf.com/{ws.get("href")}'
    ).text
    ws_soup = BeautifulSoup(html_content, "html.parser")

    papers_elements = ws_soup.find_all(
        "a", attrs={"href": re.compile(r"/content/ICCV2025W/.*.html")}
    )

    titles = [p.text for p in papers_elements]
    links = [p.get("href") for p in papers_elements]
    authors = []
    
    for element in papers_elements:
        
        dd = element.find_next("dd")
        all_forms = dd.find_all("form", recursive=False)
        paper_authors = []

        for form in all_forms:
            author = form.input.get("value")
            print(author)
            paper_authors.append(author)
        authors.append(str(paper_authors))

    all_titles += titles
    all_authors += authors
    all_links += links

Yubeen Lee
Sangeun Lee
Chaewon Park
Junyeop Cha
Eunil Park
Benjamin Greenberg
Michael Grossi
Sai Likhith Karri
Jingang Yi
Jacob Feldman
Karin Stromswold
Santiago Clemente
Layla Varghese
Valentina Nino
Maria Valero
Dimitrios Kollias
Stefanos Zafeiriou
Irene Kotsia
Greg Slabaugh
Damith Chamalke Senadeera
Jianian Zheng
Kaushal Kumar Keshlal Yadav
Chunchang Shao
Guanyu Hu
Elena Ryumina
Maxim Markitantov
Alexandr Axyonov
Dmitry Ryumin
Mikhail Dolgushin
Alexey Karpov
Antonyo Musabini
Jagdish Bhanushali
Rachid Benmokhtar
Victor Galizzi
Bertrand Luvison
Xavier Perrotton
Kazuki Kawamura
Nakai Kengo
Jun Rekimoto
Shuhei Tarashima
Yushan Wang
Norio Tagawa
Nam-Ho Kim
Jun-Hwa Kim
Annemarie Hoffsommer
Helen Schneider
Svetlana Pavlitska
Marius Zöllner
Konstantinos Spathis
Nikolaos Kardaris
Petros Maragos
Xinyi Ni
Zijian Wu
Lu Liu
Siyang Song
Fatima Alghamdi
Omar Alharbi
Abdullah Aldwyish
Raied Aljadaany
Muhammad Kamran J Khan
Huda Alamri
Fadi Khatib
Dror Moran
Guy Trostianetsky
Yoni Kasten
Meirav Galu

In [25]:
all_full_links  = [base_url + e.replace("/html", "/papers").replace(".html", ".pdf") for e in all_links]

In [26]:
df = pd.DataFrame({"title": all_titles, "authors": all_authors, "paper_url": all_full_links})

In [27]:
df.iloc[0]["paper_url"]

'https://openaccess.thecvf.com/content/ICCV2025W/ABAW/papers/Lee_Dynamic_Temporal_Gating_Networks_for_Cross-Modal_Valence-Arousal_Estimation_ICCVW_2025_paper.pdf'

In [28]:
df.to_csv("../parsed_data/iccv_2025_workshops.csv")